# Day 06 · Aggregations and business metrics

Today we turn the clean lines into the numbers a business asks for: **booked revenue**, **average order value (AOV)**, **cancellation rate** and **channel share**.

**Definitions (DATA_DICTIONARY.md):**
- **Booked revenue** = the sum of clean line_net for **completed** orders only. Cancelled orders count as 0.
- **AOV** = booked revenue ÷ the number of **distinct completed orders**.
- **Cancellation rate** = cancelled orders ÷ **all** orders.

| Task | Concept |
|---|---|
| 1 | `groupBy` changes the grain: lines → orders |
| 2 | `count(*)` vs `count(col)` vs `countDistinct`, and nulls |
| 3 | Booked revenue: join the status, keep completed orders |
| 4 | Store revenue and AOV, and reconcile with the total |
| 5 | The "average of averages" trap |
| 6 | Cancellation rate with conditional aggregation |
| 7 | Channel revenue share vs order share |
| 8 | When a ratio has no answer: null, not 0 |
| 9 | The join fan-out trap |

**Needs Day 05 first:** this notebook reads `scratch/day05/clean_lines`. Run on the **pyspark-course** cluster.

In [ ]:
from pyspark.sql import functions as F
from decimal import Decimal
import json

RAW     = "/Volumes/dbw_pyspark_course/retail/raw"
SCRATCH = "/Volumes/dbw_pyspark_course/retail/scratch"

manifest = json.load(open(f"{RAW}/manifest.json"))

# Day 05 output: 767 unique lines, already with line_gross and line_net
clean_lines = spark.read.parquet(f"{SCRATCH}/day05/clean_lines")

orders = (spark.read
    .option("header", True)
    .schema("order_id STRING, customer_id STRING, store_id STRING, order_ts TIMESTAMP, status STRING, channel STRING")
    .csv(f"{RAW}/orders.csv"))

print("clean lines:", clean_lines.count())
print("orders     :", orders.count())

### Task 1 · `groupBy` changes the grain: lines → orders

**Problem:** Add up `line_net` per order to get one row per order (`order_net`), and count the lines in each order. How many orders have more than 3 lines?

**Expected:**
- **767 lines → 300 orders.** `groupBy("order_id")` decides the output grain: one row per `order_id`.
- `O0001` has `line_count = 3`.
- **79** orders have more than 3 lines.

In [ ]:
# One row per order: add up the net amount of its lines
order_totals = clean_lines.groupBy("order_id").agg(
    F.sum("line_net").alias("order_net"),
    F.count("line_id").alias("line_count"),
)

order_totals.orderBy("order_id").show(5)

print("lines :", clean_lines.count())
print("orders:", order_totals.count())
print("orders with more than 3 lines:", order_totals.filter(F.col("line_count") > 3).count())

### Task 2 · Counting, and what nulls do

**Problem:** On a small table with some nulls, compare:
- `count("*")`: counts every row
- `count("customer_id")`: counts the rows where `customer_id` is **not null**
- `countDistinct("customer_id")`: counts the **different** non-null values
- `sum("amount")`: what happens to nulls, and what if **every** value is null?

**Expected:**
- `count(*)` = **4** · `count(customer_id)` = **3** (the null is skipped) · `countDistinct` = **2** (C1 and C2).
- `sum(amount)` = **150.00**, because nulls are skipped.
- The sum of a column where **every** value is null is **null**, not 0.

In [ ]:
visits = spark.createDataFrame([
    ("V1", "C1", Decimal("100.00")),
    ("V2", "C1", Decimal("50.00")),
    ("V3", "C2", None),
    ("V4", None, None),
], "visit_id STRING, customer_id STRING, amount DECIMAL(12,2)")

visits.agg(
    F.count("*").alias("count_all_rows"),
    F.count("customer_id").alias("count_customer_id"),
    F.countDistinct("customer_id").alias("distinct_customers"),
    F.sum("amount").alias("sum_amount"),
).show()

In [ ]:
# Sum of a column where EVERY value is null -> null, not 0
visits.filter(F.col("amount").isNull()).agg(F.sum("amount").alias("sum_of_only_nulls")).show()

### Task 3 · Booked revenue

**Problem:** The order totals don't contain a status. **Join** them with `orders` to get `status`, `store_id` and `channel`. Then keep only **completed** orders and add up their totals.

**Expected:**
- The join gives **300 rows**. Both sides have one row per `order_id`, so no rows are added.
- Booked revenue = **1,829,120.00**, the same as the manifest's `booked_revenue`.
- **270** completed orders.

In [ ]:
# Put each order's status, store and channel next to its total
order_facts = order_totals.join(
    orders.select("order_id", "store_id", "status", "channel"),
    on="order_id",
    how="inner",
)
print("rows after join:", order_facts.count())

# Booked revenue: completed orders only
booked_orders = order_facts.filter(F.col("status") == "completed")

booked_revenue = booked_orders.agg(F.sum("order_net")).first()[0]
print("booked revenue  :", booked_revenue, "| manifest:", manifest["booked_revenue"])
print("completed orders:", booked_orders.count())

### Task 4 · Store revenue and AOV

**Problem:** For each store, compute revenue, the number of distinct completed orders, and **AOV = revenue ÷ orders**. Then check that the stores add up to the total, and compare AOV with the **average line value**.

**Expected:**

| store | revenue | completed_orders | aov |
|---|---|---|---|
| S01 | 387617.50 | 60 | 6460.29 |
| S02 | 355258.75 | 49 | 7250.18 |
| S03 | 354826.25 | 55 | 6451.39 |
| S04 | 385067.50 | 55 | 7001.23 |
| S05 | 346350.00 | 51 | 6791.18 |

- The sum of the store revenue equals booked revenue → `True`. Both are exact decimals, so no rounding tolerance is needed.
- Global AOV = **6774.52**.
- The average **line** value is only **2635.62**, because an order has about 2.6 lines. AOV must divide by **orders**, not lines.

In [ ]:
store_metrics = booked_orders.groupBy("store_id").agg(
    F.sum("order_net").alias("revenue"),
    F.countDistinct("order_id").alias("completed_orders"),
)
store_metrics = store_metrics.withColumn("aov", F.round(F.col("revenue") / F.col("completed_orders"), 2))

store_metrics.orderBy("store_id").show()

In [ ]:
# The stores must add up to the total
stores_total = store_metrics.agg(F.sum("revenue")).first()[0]

print("sum of store revenue:", stores_total)
print("booked revenue      :", booked_revenue)
print("equal?              :", stores_total == booked_revenue)

# Global AOV = total revenue / number of completed orders
print("global AOV          :", round(booked_revenue / booked_orders.count(), 2))

In [ ]:
# Average LINE value is not the same as average ORDER value
completed_lines = clean_lines.join(booked_orders.select("order_id"), on="order_id", how="inner")

completed_lines.agg(F.round(F.avg("line_net"), 2).alias("average_line_value")).show()

### Task 5 · The "average of averages" trap

**Problem:** Someone takes the plain average of the 5 store AOVs and calls it the company AOV. Is that right?

**Expected:**
- The plain average of the store AOVs = **6790.85**, which is **wrong**.
- Total revenue ÷ total orders = **6774.52**, the correct global AOV.

They differ because the stores have different numbers of orders. A plain average gives a 49-order store the same say as a 60-order store. Always recompute a ratio from its **totals**.

In [ ]:
# Wrong: the plain average of the 5 store AOVs
store_metrics.agg(F.round(F.avg("aov"), 2).alias("average_of_store_aovs")).show()

# Right: total revenue / total orders
store_metrics.agg(F.round(F.sum("revenue") / F.sum("completed_orders"), 2).alias("correct_global_aov")).show()

### Task 6 · Cancellation rate with conditional aggregation

**Problem:** Count completed and cancelled orders **in one pass**, then compute the cancellation rate. Its denominator is **all orders**.

`F.count(F.when(condition, 1))` counts only the rows where the condition is true. `when` gives null otherwise, and `count` skips nulls.

**Expected:** all_orders **300** · completed **270** · cancelled **30** · cancellation_rate **0.1** (10%).

In [ ]:
status_counts = orders.agg(
    F.count("*").alias("all_orders"),
    F.count(F.when(F.col("status") == "completed", 1)).alias("completed"),
    F.count(F.when(F.col("status") == "cancelled", 1)).alias("cancelled"),
)

status_counts = status_counts.withColumn("cancellation_rate", F.col("cancelled") / F.col("all_orders"))
status_counts.show()

### Task 7 · Channel revenue share vs order share

**Problem:** For each channel (`store` / `online`), compute its share of booked **revenue** and its share of completed **orders**. Check that the revenue shares add up to 1.

**Expected:**

| channel | revenue | orders | revenue_share | order_share |
|---|---|---|---|---|
| store | 1256378.75 | 180 | 0.6869 | 0.6667 |
| online | 572741.25 | 90 | 0.3131 | 0.3333 |

- The revenue shares add up to **1.0000**.
- Revenue share ≠ order share: in-store orders are a bit bigger on average, so they bring 68.7% of the money from 66.7% of the orders.

In [ ]:
channel_revenue = booked_orders.groupBy("channel").agg(
    F.sum("order_net").alias("revenue"),
    F.count("order_id").alias("orders"),
)

completed_order_count = booked_orders.count()

channel_share = (channel_revenue
    .withColumn("revenue_share", F.round(F.col("revenue") / F.lit(booked_revenue), 4))
    .withColumn("order_share",   F.round(F.col("orders") / F.lit(completed_order_count), 4)))

channel_share.show()
channel_share.agg(F.sum("revenue_share").alias("total_revenue_share")).show()

### Task 8 · When a ratio has no answer: null, not 0

**Problem:** A new store `S99` has **no completed orders** yet. What is its AOV?

**Expected:**
- `S01` → 250.00.
- `S99` → **null**. An AOV of 0 would claim "its orders were worth ₹0 on average", but there were **no orders** to average. Null means "not defined".
- Write the rule explicitly with `when`, so it doesn't depend on Spark settings.

In [ ]:
stores_test = spark.createDataFrame([
    ("S01", Decimal("1000.00"), 4),
    ("S99", Decimal("0.00"),    0),     # new store, no completed orders yet
], "store_id STRING, revenue DECIMAL(12,2), completed_orders INT")

stores_test.withColumn("aov",
    F.when(F.col("completed_orders") == 0, None)                      # no orders -> undefined
     .otherwise(F.col("revenue") / F.col("completed_orders"))
).show()

### Task 9 · The join fan-out trap

**Problem:** Someone joins the **order totals** back onto the **lines**, then adds up `order_net`. What happens?

**Expected:**
- **694 rows**, one per completed line, and each order's total is now **repeated on every one of its lines**.
- The sum is **5,552,532.50** instead of 1,829,120.00, about **3×** too much.

The rule: add up a value at **the grain where it lives**. `order_net` belongs to orders, so sum it over orders, not lines.

In [ ]:
# Join the order totals back onto the lines: each total is now repeated on every line of its order
lines_with_order_total = completed_lines.join(booked_orders.select("order_id", "order_net"), on="order_id")

print("rows:", lines_with_order_total.count())
lines_with_order_total.agg(F.sum("order_net").alias("inflated_revenue")).show()

### Key takeaways
- **`groupBy` sets the output grain:** one row per group key, so lines → orders.
- **Counting:**
  - `count("*")` counts rows.
  - `count(col)` skips nulls.
  - `countDistinct` counts the different values.
  - `sum` skips nulls, and gives **null** if every value is null.
- **Booked revenue = completed orders only.** Join the status in, then filter.
- **AOV = revenue ÷ distinct orders**, never ÷ lines. Average line value is a different metric.
- **Never average averages.** Recompute ratios from totals (Task 5).
- **Denominators matter:** the cancellation rate uses **all** orders.
- **Conditional aggregation:** `count(when(cond, 1))` counts many categories in one pass.
- **A share of money ≠ a share of orders.**
- **Undefined ratio → null**, not 0.
- **Fan-out:** joining a higher-grain total (order) onto lower-grain rows (lines) repeats it, and summing then inflates the result.
- **Reconcile:** the parts (stores) must add up exactly to the whole (booked revenue).

*Skipped on purpose (they don't add a new concept): top product by units vs money, approximate distinct counts, shuffle inspection, and the metric-definition document.*